## Telco Customer Churn Prediction & Machine Learning Pipeline

metadata here

note: synthetic data for notes purposes

### 1. Project Setup & Synthetic Data Generator

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML libraries (Scikit-Learn)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import(
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

In [6]:
sns.set_theme(style="whitegrid")

# generate synthetic telco churn dataset for e2e ds pipeline
np.random.seed(42) # starting point for generating pseudo-random numbers, making code completely reproducible
n_samples = 2000 # there will be 2000 number of data points (rows/observations)

# creating dictionary first to create synthetic data
data = {
    'tenure_months': np.random.randint(1, 72, n_samples),
    'monthly_charges': np.random.uniform(20.0, 120.0, n_samples),
    'total_charges': None, # To be engineered
    'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.5, 0.3, 0.2]),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.4, 0.4, 0.2]),
    'payment_method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples),
    'senior_citizen': np.random.choice([0, 1], n_samples, p=[0.8, 0.2])
}

df_churn = pd.DataFrame(data) # convert data dictionary to dataframe

# calculations for total charges
df_churn['total_charges'] = df_churn['tenure_months'] * df_churn['monthly_charges'] + np.random.normal(0, 50, n_samples)
df_churn['total_charges'] = df_churn['total_charges'].clip(lower=20.0) # sets a floor of 20.0 values (values < 20.0 become 20.0) so that it won't produce negative charge or unrealistic low value

# simulate churn probability based on features (ground truth rule)
# this creates a realistic math formula that calculates each customer's likelihood of cancelling their service based on their account details
churn_prob = (
    0.35 * (df_churn['contract_type'] == 'Month-to-month') +
    0.30 * (df_churn['internet_service'] == 'Fiber optic') +
    0.25 * (df_churn['payment_method'] == 'Electronic check') -
    0.01 * df_churn['tenure_months'] +
    0.003 * df_churn['monthly_charges']
)

# runs the raw `churn_prob` numbers through the sigmoid function, squeezing all values strictly into a valid 0.0 to 1.0 (0% to 100%) probability range
# why? previous linear addition step could produce numbers less than 0 or greather than 1
# sigmoid func creates as 'S-shaped' curve that smoothly maps negative values close to 0 and large positive values close to 1, turning abstract math scores into true mathematically probabilities
churn_prob = 1 / (1 + np.exp(-churn_prob)) # Sigmoid transformation
# print(churn_prob) to see before transforming 'churn' as what the next line (below this comment) states

# this performs a weighted 'coin flip' for every customer based on individual probability score
# ^ assigns them a final status of 1 (churned) or 0 (retained)
df_churn['churn'] = (np.random.binomial(1, churn_prob) == 1).astype(int)

print(f"Dataset Shape: {df_churn.shape[0]} rows, {df_churn.shape[1]} columns.")
df_churn.head()

Dataset Shape: 2000 rows, 8 columns.


,tenure_months,monthly_charges,total_charges,contract_type,internet_service,payment_method,senior_citizen,churn
0,52,45.467062,2334.270302,One year,No,Bank transfer,0,0
1,15,104.087158,1590.984341,Two year,DSL,Electronic check,0,0
2,61,23.842635,1454.553735,Month-to-month,Fiber optic,Electronic check,0,1
3,21,110.176199,2342.931331,Month-to-month,Fiber optic,Mailed check,1,1
4,24,66.147746,1599.437392,Month-to-month,Fiber optic,Mailed check,0,1
